# 01. Какие данные есть в репозитории

Перед тем как лезть в трекеры, разбираемся **на чём они вообще работают**.

В репо две группы данных:

1. **Синтетика** в `generated_dataset/` — 4 короткие пары `score.json` + `performance.mid` (8–13 нот). Это unit-тестовый уровень, удобный для разбора алгоритмов по шагам.
2. **Реальные пьесы** в `midi/library/` — большие партитуры (Рахманинов `rach_solo`, 3791 score-state), но **без отдельных записей живого исполнения**. То есть это только нотный текст.

Чтобы обучать RL-агента по тезисам, нужен корпус **исполнений с разметкой нота-в-ноту** — ASAP / MAESTRO. Их импортёр будет в `musictech/datasets/`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import mido

print("Project root:", PROJECT_ROOT)

Project root: d:\Projects-13-03-2026\piano


## 1. Что в `generated_dataset/`

In [2]:
synthetic_dir = PROJECT_ROOT / "generated_dataset"

for json_path in sorted(synthetic_dir.glob("*.json")):
    midi_path = json_path.with_suffix(".mid")
    
    score = json.loads(json_path.read_text(encoding="utf-8"))
    midi_events = sum(
        1 for msg in mido.MidiFile(midi_path) if msg.type == "note_on" and msg.velocity > 0
    )
    print(
        f"{json_path.name:18s} score_notes={len(score['notes']):3d} "
        f"perf_events={midi_events:3d} ({midi_path.name})"
    )

ideal.json         score_notes=  8 perf_events=  8 (ideal.mid)
noisy.json         score_notes=  8 perf_events= 14 (noisy.mid)
polyphonic.json    score_notes= 13 perf_events= 13 (polyphonic.mid)
rubato.json        score_notes=  8 perf_events=  8 (rubato.mid)


## 2. Что в `midi/library/` (реальные пьесы)

Library пьесы создаются скриптом `midi_workspace.py` из произвольного MIDI. Структура одного workspace:

```
midi/library/<slug>/
├── source/source.{mid, json, hybrid_profile.json}     # полная партитура
├── study_mode/{left, right}_hand.{mid, json, ...}     # отдельные руки
├── orchestra/orchestra.mid                            # аккомпанемент
└── workspace.json                                      # метаданные
```

In [ ]:
library_dir = PROJECT_ROOT / "midi" / "library"
manifest = json.loads((library_dir / "manifest.json").read_text(encoding="utf-8"))

for piece in manifest["pieces"]:
    print(f"=== {piece['title']} ({piece['slug']}) ===")
    print(f"  full score states: {piece['source']['full_score_states']}")
    sm = piece.get("study_mode", {})
    
    print(
        f"  left/right notes : {sm.get('left_notes')}/{sm.get('right_notes')}"
    )
    print(f"  orchestra MIDI   : {piece['orchestra']['imported_midi']}")
    print()

=== rach_solo (rach_solo) ===
  full score states: 3791
  left/right notes : 3552/3604
  orchestra MIDI   : midi/library/rach_solo/orchestra/orchestra.mid
=== rach_solo (rach_solo_2) ===
  full score states: 3791
  left/right notes : 3552/3604
  orchestra MIDI   : midi/library/rach_solo_2/orchestra/orchestra.mid


## 3. Чего не хватает для тезисов

Для **обучения** RL-агента нужен корпус, где для каждой score-state известно, в какой момент времени исполнитель её сыграл. В репо такого нет — есть только синтетика и сами партитуры.

Кандидаты на импорт (см. `ARCHITECTURE.md` § 5.2):

- **ASAP** (Foscarin et al. 2020): 222 пьесы, 1067 исполнений, классическое фортепиано, есть `note_alignments/note_alignment.tsv` с note-by-note маппингом между score MIDI и performance MIDI.
- **MAESTRO v3**: 1276 файлов с Disklavier, аудио + MIDI, без явного score↔performance alignment (но можно сводить через DTW).

ASAP импортируется в `datasets/asap/<piece>/{score.json, performances/<id>.json}` — это будет следующий шаг.